# RAG Retrieval Experiments with LlamaIndex

## Objective
Builds a vector index over a mortgage/loan document and compares retrieval settings such as top-k, score thresholds, and reranking placeholders.

## Approach
- Load the PDF using LlamaIndex
- Create embeddings with BGE small
- Run retrieval experiments with different configurations
- Generate answers from retrieved context

## Expected Result
This notebook documents retrieval tuning for a mortgage Q&A pipeline, showing how retrieval settings affect context quality and answer reliability.

## Running in Google Colab
These notebooks were developed in Google Colab. For reproducibility, place required PDFs in a `data/` folder when running locally, or upload them to the Colab working directory. The helper functions below try common Colab and GitHub-style paths.

## Security Note
API keys are not stored in the notebook. Use Colab Secrets with the name `GOOGLE_API_KEY` or set the environment variable manually.

## Project Context
This notebook is part of a curated document intelligence externship portfolio project completed through Outamation. The work focuses on OCR, document parsing, retrieval, LLM-based question answering, and prototype application development for mortgage-style document analysis.

## Data Note
The notebooks were originally developed in Google Colab. Any document files used for testing should be placed in the `data/` folder or uploaded directly into the Colab runtime. The sample documents used for this educational project do not contain sensitive personal information.


In [ ]:
# -------------------------
# PORTABLE COLAB/GITHUB HELPERS
# -------------------------
from pathlib import Path
import os

def resolve_path(filename_or_path):
    """Find a file in common Colab and GitHub project locations."""
    candidates = [
        Path(filename_or_path),
        Path("/content") / filename_or_path,
        Path("data") / Path(filename_or_path).name,
        Path("/content/data") / Path(filename_or_path).name,
    ]
    for path in candidates:
        if path.exists():
            return str(path)
    # Return GitHub-style path as the default so users know where to place data.
    return str(Path("data") / Path(filename_or_path).name)

def get_google_api_key():
    """Load GOOGLE_API_KEY from Colab Secrets or environment variables."""
    try:
        from google.colab import userdata
        key = userdata.get("GOOGLE_API_KEY")
        if key:
            os.environ["GOOGLE_API_KEY"] = key
            return key
    except Exception:
        pass
    return os.getenv("GOOGLE_API_KEY")

!pip -q install --upgrade \
  llama-index \
  llama-index-embeddings-huggingface \
  llama-index-llms-google-genai \
  llama-index-readers-file \
  sentence-transformers \
  pypdf \
  nest_asyncio
import nest_asyncio
nest_asyncio.apply()
import os

from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

GEMINI_API_KEY = get_google_api_key()
if not GEMINI_API_KEY:
    raise ValueError("GOOGLE_API_KEY not found. Add it in Colab Secrets or set it as an environment variable.")

Settings.llm = GoogleGenAI(model="gemini-2.5-flash", api_key=GEMINI_API_KEY)
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
Settings.text_splitter = SentenceSplitter(chunk_size=512, chunk_overlap=50)

documents = SimpleDirectoryReader(
    input_files=[resolve_path("LenderFeesWorksheetNew.pdf")]
).load_data()

index = VectorStoreIndex.from_documents(documents)

print("Index built.")
def run_experiment(index, query, top_k=5, apply_threshold=False, threshold=0.75, use_reranker=False):
    retriever = index.as_retriever(similarity_top_k=top_k)
    nodes = retriever.retrieve(query)

    # optional threshold filter
    if apply_threshold:
        filtered_nodes = []
        for node in nodes:
            score = getattr(node, "score", None)
            if score is not None and score > threshold:
                filtered_nodes.append(node)
        nodes = filtered_nodes

    # optional reranking placeholder
    # If you already used a reranker in class, plug it in here.
    # For now this code keeps the same nodes unless you add a reranker object.
    final_nodes = nodes

    print("=" * 100)
    print(f"QUERY: {query}")
    print(f"top_k={top_k}, threshold={'on' if apply_threshold else 'off'}, "
          f"threshold_value={threshold if apply_threshold else 'N/A'}, reranker={'on' if use_reranker else 'off'}")
    print("=" * 100)

    print(f"\nRetrieved {len(final_nodes)} chunks.\n")

    for i, node in enumerate(final_nodes, 1):
        score = getattr(node, "score", None)
        if score is not None:
            print(f"\nChunk {i} (Score: {score:.4f})")
        else:
            print(f"\nChunk {i}")
        print(node.get_text()[:1500])

    context = "\n\n".join([node.get_text() for node in final_nodes])

    prompt = f"""
Use only the context below to answer the question.

Context:
{context}

Question:
{query}

Answer clearly and briefly.
"""

    response = Settings.llm.complete(prompt)

    print("\n" + "-" * 100)
    print("FINAL GENERATED ANSWER:")
    print(str(response))
    print("-" * 100)

    best_chunk = final_nodes[0].get_text()[:200] if final_nodes else "No chunk passed filtering."
    answer_text = str(response)

    return {
        "chunks_retrieved": len(final_nodes),
        "best_chunk_excerpt": best_chunk,
        "answer": answer_text
    }


# Default evaluation query used by the retrieval experiments below.
query = "How much does the borrower pay for lender\'s title insurance?"


In [ ]:
result_A = run_experiment(
    index=index,
    query=query,
    top_k=5,
    apply_threshold=False,
    threshold=0.75,
    use_reranker=False
)

# Default evaluation query used by the retrieval experiments below.
query = "How much does the borrower pay for lender\'s title insurance?"


In [ ]:
result_B = run_experiment(
    index=index,
    query=query,
    top_k=8,
    apply_threshold=True,
    threshold=0.75,
    use_reranker=False
)

# Default evaluation query used by the retrieval experiments below.
query = "How much does the borrower pay for lender\'s title insurance?"


In [ ]:
result_C = run_experiment(
    index=index,
    query=query,
    top_k=5,
    apply_threshold=True,
    threshold=0.75,
    use_reranker=True
)


# Default evaluation query used by the retrieval experiments below.
query = "How much does the borrower pay for lender\'s title insurance?"
